# Crime Type Classification
Can we guess which type of crime happened just from where, when, and how it happened? Predicting the top 5 crime categories using area, time of day, day of week, hour, victim sex, and whether a weapon was involved.

In [ ]:
# libraries for this project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# fixing the random state everywhere so results dont change every run
RANDOM_STATE = 42

In [ ]:
import sklearn, platform, time
print("python:", platform.python_version())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
# also written out in requirements.txt, printed here too so it shows up
# in the executed notebook itself, not just a separate file

## 1. Load and audit the data
First check the raw shape, types, and what's missing before doing anything to it.

In [ ]:
df = pd.read_csv("./Data/Crime_Data.csv")
print("raw shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# missing values per column
df.isna().sum()

In [ ]:
# how many duplicate rows are there
df.duplicated().sum()

## 2. Cleaning
Same cleaning approach as the earlier exploratory notebook, just documented again here since this is a separate workflow.

In [ ]:
# drop rows with no occurrence date since we need real dates for the time features
df = df[~df["DATE OCC"].isna()]

# clean up the messy sex labels, put missing/unknown ones into their own category
# instead of dropping them, since sex is one of our model features
df["VICT SEX"] = df["VICT SEX"].str.upper().str.strip()
df["VICT SEX"] = df["VICT SEX"].replace({"UNKNOWN": np.nan, "X": np.nan})
df["VICT SEX"] = df["VICT SEX"].fillna("UNKNOWN")

# same idea for weapon, missing almost certainly means no weapon was used
df["WEAPON"] = df["WEAPON"].fillna("WEAPON UNKNOWN")

print("shape after cleaning:", df.shape)

## 3. Feature engineering
Building the columns we'll actually feed into the models.

In [ ]:
df["DATE OCC"] = pd.to_datetime(df["DATE OCC"], errors="coerce")
df["TIME OCC"] = pd.to_numeric(df["TIME OCC"], errors="coerce")
df["HOUR OCC"] = (df["TIME OCC"] // 100).astype("Int64")
df["DAY OF WEEK"] = df["DATE OCC"].dt.day_name()

def time_of_day(h):
    if pd.isna(h):
        return np.nan
    h = int(h)
    if 5 <= h < 12:
        return "Morning"
    if 12 <= h < 17:
        return "Afternoon"
    if 17 <= h < 21:
        return "Evening"
    return "Night"

df["TIME OF DAY"] = df["HOUR OCC"].apply(time_of_day)

# turning weapon into a simple yes/no flag instead of the full messy weapon list,
# a lot easier for the model to use than 50+ weapon categories
df["HAS WEAPON"] = (df["WEAPON"] != "WEAPON UNKNOWN").astype(int)

df[["HOUR OCC", "DAY OF WEEK", "TIME OF DAY", "HAS WEAPON"]].head()

## 4. Defining the classification target
Predicting every single crime type would give us dozens of tiny classes, so we're keeping it to the 5 most common crime types. That still covers a meaningful chunk of the dataset and keeps the classes big enough to actually learn from.

In [ ]:
top5 = list(df["CRIME"].value_counts().head(5).index)
print("target classes:")
for c in top5:
    print("-", c)

df_model = df[df["CRIME"].isin(top5)].copy()
print("\nrows kept for modeling:", df_model.shape[0], "out of", df.shape[0])

In [ ]:
feature_cols = ["AREA NAME", "TIME OF DAY", "DAY OF WEEK", "HOUR OCC", "HAS WEAPON", "VICT SEX"]
target_col = "CRIME"

# drop any rows still missing a value in the columns we actually use
df_model = df_model.dropna(subset=feature_cols + [target_col])
df_model["HOUR OCC"] = df_model["HOUR OCC"].astype(int)

print("rows after dropping feature NaNs:", df_model.shape[0])

## 4b. Data dictionary
Only the columns actually used are documented here in detail (the full raw
schema has 11 columns; most either aren't predictive of crime type on their
own, e.g. free-text `LOCATION`, or duplicate information already captured by
the engineered features).

| Column | Type | Values / Range | Role | Notes |
|---|---|---|---|---|
| `CRIME` | categorical (str) | 5 classes after filtering (see above) | **target** | Restricted to the 5 most frequent crime types out of 131 raw categories |
| `AREA NAME` | categorical (str) | 21 LAPD patrol areas | feature | e.g. "Central", "Southeast"; encoded with one-hot |
| `TIME OF DAY` | categorical (str), derived | Morning / Afternoon / Evening / Night | feature | Bucketed from `HOUR OCC` |
| `DAY OF WEEK` | categorical (str), derived | Monday ... Sunday | feature | Extracted from `DATE OCC` |
| `HOUR OCC` | numeric (int), derived | 0-23 | feature | `TIME OCC` (HHMM) // 100 |
| `HAS WEAPON` | numeric (binary), derived | 0 / 1 | feature | 1 if a weapon was logged, 0 if `WEAPON` was missing |
| `VICT SEX` | categorical (str) | F / M / UNKNOWN | feature | Blanks, `X`, and `UNKNOWN` collapsed into a single `UNKNOWN` category |

Columns present in the raw file but **not** used as model inputs: `DATE RPTD`,
`DATE OCC` (used only to derive `DAY OF WEEK`), `TIME OCC` (used only to derive
`HOUR OCC`), `DIST NO` (finer-grained duplicate of `AREA`), `PREMISES` (not
used in this pass - see limitations), `WEAPON` (collapsed to the `HAS WEAPON`
flag instead of using the full ~50-category list), and `LOCATION` (free-text
street address, too high-cardinality to encode directly).

## 5. Exploratory visuals
A look at the target class balance and one feature relationship before modeling.

In [ ]:
class_counts = df_model["CRIME"].value_counts()

plt.figure(figsize=(9, 5))
class_counts.plot(kind="bar", color="teal")
plt.title("Class Distribution: Top 5 Crime Types")
plt.ylabel("Number of incidents")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(class_counts / class_counts.sum())

In [ ]:
# does crime type shift depending on time of day? this is the feature relationship
# that motivated including TIME OF DAY as a predictor in the first place
time_crime = df_model.groupby(["TIME OF DAY", "CRIME"]).size().unstack()
time_crime = time_crime[top5]

time_crime.plot(kind="bar", stacked=True, figsize=(10, 5), colormap="tab20")
plt.title("Crime Type by Time of Day")
plt.ylabel("Number of incidents")
plt.legend(title="Crime type", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

The class counts show real imbalance right away, vehicle theft alone is more than a third of these five categories, and theft of identity is the smallest at around 14%. That imbalance is something we need to watch for when picking metrics later (accuracy alone would hide how badly a model does on the smaller classes).

## 6. Train / validation / test split
Splitting before doing any preprocessing so nothing from validation or test leaks into how we clean or scale the training data. Using stratified splits since the classes are imbalanced, want each split to keep roughly the same class ratios.

In [ ]:
X = df_model[feature_cols]
y = df_model[target_col]

# first pull out a test set that we will not touch again until the very end
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# then split the remaining 80% into train and validation, used only for tuning
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_STATE, stratify=y_trainval
)  # 0.25 of the remaining 80% works out to 60/20/20 overall

print("train:", X_train.shape)
print("val:  ", X_val.shape)
print("test: ", X_test.shape)

## 7. Preprocessing pipeline
One-hot encoding for the categorical columns and scaling for the numeric ones, wrapped in a pipeline so the same transformation gets fit only on training data and then reused consistently on validation and test.

In [ ]:
categorical_cols = ["AREA NAME", "TIME OF DAY", "DAY OF WEEK", "VICT SEX"]
numeric_cols = ["HOUR OCC", "HAS WEAPON"]

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numeric_cols),
])

## 8. Baseline
Before trying any real model, set a floor with the simplest possible rule: always guess the majority class.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
base_val_pred = baseline.predict(X_val)

print("baseline val accuracy:", accuracy_score(y_val, base_val_pred))
print("baseline val macro f1:", f1_score(y_val, base_val_pred, average="macro"))

The baseline's macro F1 is really low, which makes sense: it only ever picks one class, so it gets zero right on the other four. Any model we build needs to clear this bar by a wide margin to be worth using.

### Why these three models
- **Logistic Regression** is the natural starting point above the baseline: it's
  linear, fast, and gives directly interpretable coefficients per class, which
  matters for a criminal-justice-adjacent use case where "why did the model say
  this" is a real question. It needs scaled numeric features and encoded
  categoricals, both of which the pipeline already provides.
- **K-Nearest Neighbors** tests whether *local* similarity (calls that happened
  at a similar hour, area, and day) is more informative than a single global
  linear boundary. It's more sensitive to feature scaling and to the curse of
  dimensionality from one-hot encoding a 21-category area column, which is why
  k is tuned over a wider range than usual (5, 15, 25) - small k overfits to
  individual encoded corners.
- **Random Forest** tests whether the target is being driven by *interactions*
  between features (e.g., "night + Central + weapon" behaving differently
  from what any single feature would predict). It doesn't need scaling and
  handles the mixed categorical/numeric feature set natively, at the cost of
  being harder to interpret directly than logistic regression.

All three are taught, comparably cheap to train on ~58K rows, and different
enough in their assumptions (linear vs. local vs. tree-based) that comparing
them says something about the *structure* of the problem, not just which
library implementation happens to win.

## 9. Model tuning on the validation set
Trying a small number of hyperparameter settings per model and picking whichever does best on the validation set. Not touching the test set at all during this step.

In [ ]:
best_logreg = None
for C in [0.1, 1.0, 10.0]:
    pipe = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, C=C, random_state=RANDOM_STATE))])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    f1 = f1_score(y_val, pred, average="macro")
    print(f"logistic regression C={C}: val macro f1 = {f1:.4f}")
    if best_logreg is None or f1 > best_logreg[0]:
        best_logreg = (f1, C)

best_C = best_logreg[1]
print("\nbest C:", best_C)

In [ ]:
best_knn = None
for k in [5, 15, 25]:
    pipe = Pipeline([("prep", preprocess), ("clf", KNeighborsClassifier(n_neighbors=k))])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    f1 = f1_score(y_val, pred, average="macro")
    print(f"knn k={k}: val macro f1 = {f1:.4f}")
    if best_knn is None or f1 > best_knn[0]:
        best_knn = (f1, k)

best_k = best_knn[1]
print("\nbest k:", best_k)

In [ ]:
rf_configs = [
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 100, "max_depth": None},
    {"n_estimators": 200, "max_depth": 15},
]

best_rf = None
for cfg in rf_configs:
    pipe = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(random_state=RANDOM_STATE, **cfg))])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    f1 = f1_score(y_val, pred, average="macro")
    print(f"random forest {cfg}: val macro f1 = {f1:.4f}")
    if best_rf is None or f1 > best_rf[0]:
        best_rf = (f1, cfg)

best_rf_cfg = best_rf[1]
print("\nbest random forest config:", best_rf_cfg)

Worth noting: the fully-grown random forest (`max_depth=None`) actually did *worse* on validation than the shallower ones. That's overfitting, an unrestricted forest memorizes the training data instead of learning a pattern that generalizes, so we go with the shallower, better-generalizing config instead of the most "powerful" looking one.

## 10. Final evaluation on the untouched test set
Now that hyperparameters are locked in from the validation results, refit each model on train+validation combined and evaluate once on the test set. No more decisions get made after seeing these numbers.

In [ ]:
final_models = {
    "Baseline (majority class)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, C=best_C, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=best_k),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, **best_rf_cfg),
}

results = {}
test_predictions = {}
test_probabilities = {}

for name, clf in final_models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])

    fit_start = time.time()
    pipe.fit(X_trainval, y_trainval)
    fit_time = time.time() - fit_start

    pred_start = time.time()
    pred = pipe.predict(X_test)
    pred_time = time.time() - pred_start

    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
        "fit_time_sec": fit_time,
        "predict_time_sec": pred_time,
    }
    test_predictions[name] = pred
    if hasattr(pipe, "predict_proba"):
        test_probabilities[name] = (pipe.predict_proba(X_test), pipe.classes_)

results_df = pd.DataFrame(results).T
results_df


In [ ]:
results_df[["accuracy", "macro_f1"]].plot(kind="bar", figsize=(9, 5))
plt.title("Model Comparison on Test Set")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

Training time is worth reporting alongside accuracy: a model that's a
fraction of a percent better but many times slower to train and refit is a
real trade-off, not a free upgrade.

In [ ]:
results_df[["fit_time_sec", "predict_time_sec"]].plot(kind="bar", figsize=(9, 5))
plt.title("Computational Cost by Model (test-set refit)")
plt.ylabel("Seconds")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

print(results_df[["accuracy", "macro_f1", "fit_time_sec", "predict_time_sec"]])

## 11. Diagnostics for the best model
Logistic regression comes out ahead on macro F1, so that's our selected model. Looking closer at where it gets things right and wrong.

In [ ]:
best_model_name = results_df["macro_f1"].idxmax()
print("selected model:", best_model_name)

print(classification_report(y_test, test_predictions[best_model_name]))

In [ ]:
cm = confusion_matrix(y_test, test_predictions[best_model_name], labels=top5)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")

short_labels = [c[:18] for c in top5]
ax.set_xticks(range(len(top5)))
ax.set_yticks(range(len(top5)))
ax.set_xticklabels(short_labels, rotation=45, ha="right")
ax.set_yticklabels(short_labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix - {best_model_name}")

for i in range(len(top5)):
    for j in range(len(top5)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im)
plt.tight_layout()
plt.show()

### Macro ROC-AUC
The classification report already gives class-wise precision/recall/F1;
one-vs-rest macro ROC-AUC adds a threshold-independent view of how well the
model *ranks* the correct class above the others, which is useful since the
five classes are imbalanced (see Section 5).

### Sanity-checking macro-F1 by hand with NumPy
`classification_report` already gives per-class and macro F1, but it's easy to
just trust a library number without knowing what it's doing. Recomputing it
directly from the confusion matrix with NumPy (no sklearn metric functions)
is a quick way to double check it, and it's a genuinely useful thing to be
able to explain during the viva.

In [ ]:
cm_full = confusion_matrix(y_test, test_predictions[best_model_name], labels=top5)

# for each class: true positives are the diagonal, false positives are the
# rest of that column, false negatives are the rest of that row
tp = np.diag(cm_full).astype(float)
fp = cm_full.sum(axis=0) - tp
fn = cm_full.sum(axis=1) - tp

precision_manual = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) != 0)
recall_manual = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) != 0)
f1_manual = np.divide(
    2 * precision_manual * recall_manual,
    precision_manual + recall_manual,
    out=np.zeros_like(tp),
    where=(precision_manual + recall_manual) != 0,
)

manual_table = pd.DataFrame(
    {"precision": precision_manual, "recall": recall_manual, "f1": f1_manual},
    index=top5,
)
print(manual_table.round(3))
print("\nmanual macro F1:", round(f1_manual.mean(), 4))
print("sklearn macro F1 (for comparison):",
      round(f1_score(y_test, test_predictions[best_model_name], average="macro"), 4))

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score

if best_model_name in test_probabilities:
    proba, proba_classes = test_probabilities[best_model_name]
    y_test_bin = label_binarize(y_test, classes=list(proba_classes))
    macro_roc_auc = roc_auc_score(y_test_bin, proba, average="macro", multi_class="ovr")
    print(f"{best_model_name} macro ROC-AUC (one-vs-rest): {macro_roc_auc:.4f}")
else:
    print(f"{best_model_name} does not expose predict_proba; skipping ROC-AUC.")

The confusion matrix shows vehicle theft is the easiest class to get right (mostly along the diagonal), which lines up with it being the largest and most distinct class. The three theft-related categories get confused with each other the most, which makes sense since "shoplifting", "petty theft" and "identity theft" likely share similar circumstances (same areas, similar times) that our features can't tell apart. This is the imbalance and feature-limitation problem showing up directly in the errors, not something a fancier model would necessarily fix.

## 12. Limitations, Assumptions, and Responsible Use
- The feature set is fairly shallow (area, time, day, sex, weapon flag); it doesn't capture things like exact location, suspect description, or premises type, which likely explains a lot of the confusion between similar theft categories.
- Classes are imbalanced (vehicle theft is over 2x the size of theft of identity), which macro F1 accounts for but still limits what any model can learn about the smaller classes.
- This is a random stratified split rather than a time-based split, so the model isn't being tested on how well it generalizes to *future* time periods, only to unseen rows from the same time range. If crime patterns shift over time (seasonally, or due to policy changes), performance on genuinely future data could be lower than the test-set numbers here suggest.
- **`AREA NAME` is a proxy, not a cause.** A patrol area correlates with crime type through a mix of actual crime patterns *and* historical patrolling and reporting intensity in that area. The model cannot separate "more crime happens here" from "more crime gets reported/recorded here," so area-based predictions risk reinforcing whatever reporting bias already exists in the data rather than describing crime itself.
- **`VICT SEX` is included as a feature** because it's genuinely associated with certain crime types in this data, but that association is descriptive of the historical dataset, not a claim that sex causes or predicts victimization in general. Using it operationally (e.g., to change how a call is prioritized) would need separate fairness scrutiny well beyond what this project does.
- None of this should be read as causal. A high-crime area or time window doesn't mean the model understands *why* crime happens there, only that it found a statistical association in this dataset.
- **What this model should not be used for:** it is not suitable for deciding individual-level suspicion, allocating patrols in a way that could compound reporting bias in already over-patrolled areas, or any use where a wrong prediction has a real consequence for a specific person. At best it is a coarse, aggregate description of how *recorded* crime types have historically distributed across time and area in this one dataset.